# **EDA & Data Perparation**


**Covering:**

* Data verification
* Cleaning
* Profit/Margin
* Inventory-linked risk analysis
* validation check against the known stockout/slow-mover labels

 ## **Day 01**

**Loading Libraries**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import os

import warnings
warnings.filterwarnings("ignore")

**Loading Dataset**

In [ ]:

# **Day 01 - Data Preparation & EDA (Updated: FORESIGHT dataset)**from collections import defaultdict
import kagglehub

path = kagglehub.dataset_download("mrayyanshehzad/synthetic-retail-dataset-10-million-transactions")

print("Path to dataset files:", path)


files = os.listdir(path)
print("\nParent Folder contain files:", files)
# Update File Name to select which File to Load
if 'retail_contaminated_dataset' in files:
    path = os.path.join(path, 'retail_contaminated_dataset')
    files = os.listdir(path)
    print("Sub-folder contain files:", files)

csv_files = []

for f in files:
  if f.endswith(".csv"):
     csv_files.append(f)

print("\nCSV files in the folder:", csv_files)


df = {}

for file in csv_files:
    file_path = os.path.join(path, file)

    df_name = file.replace('.csv', '')
    df[df_name] = pd.read_csv(file_path)
    print(f"Loaded: {file} as dataframes['{df_name}'] (Shape: {df[df_name].shape})")

**Dataset Loading Flow**

```
📁 /kaggle/input/synthetic-retail-dataset-10-million-transactions (Main Path)
│
├── 📑 README.md
└── 📁 Synthetic Retail Dataset/  <-- [Line: if 'Synthetic Retail Dataset' in files]
    │                                  Enter inside this folder to access .csv files
    ├── 📄 sales_transactions.csv
    ├── 📄 sku_inventory_flags.csv
    ├── 📄 inventory_snapshot.csv
    ├── 📄 store_master.csv
    ├── 📄 customer_master.csv
    ├── 📄 sku_master.csv
    └── 📄 promotions.csv
    
```

**Dataset Relationship**
```
store_master.store_id      ──┐
sku_master.sku_id           ─┼──▶ sales_transactions.csv
customer_master.cust_id     ─┤     (store_id, sku_id, customer_id, promo_id)
promotions.promo_id         ─┘

store_master.store_id   ──┐
sku_master.sku_id        ─┴──▶ inventory_snapshot.csv

sku_master.sku_id  ──▶ sku_inventory_flags.csv   (GROUND-TRUTH labels — do NOT use as a feature,
                                                    only to validate our own risk logic later)
```

### **Data View, Shape & Columns**

In [ ]:
for name, table in df.items():
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    print(f" {name}")
    print("---------------------------------------------------------------------------------------------------------------------------------------------------------------------------")
    display(table.head(3))
    print("shape:", table.shape)
    print("columns:", list(table.columns))
    print()

### **Step 1 — Verification (before manipulating data)**


**Primary Data Verification**

In [ ]:
expected_rows = {
    "store_master": 30,
    "sku_master": 5000,
    'customer_master': 10000,
    'promotions': 100,
}

for name, expected in expected_rows.items():
    actual = len(df[name])
    # status = "OK" if actual == expected else "MISMATCH"
    status = "Unknown"
    if actual == expected:
      status = "OK"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}   [{status}]")
    elif actual > expected:
      status = "EXCESS"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}  [{status}]")
    else:
      status = "MISMATCH"
      print(f"{name:20s}   actual={actual:>7}   expected={expected:>7}  [{status}]")

**Secondary Data Verification**

In [ ]:
print(f"{'inventory_snapshot':20s}   actual={len(df['inventory_snapshot']):>8}   expected= no fixed target (quick-check only)")
print(f"{'sales_transactions':20s}   actual={len(df['sales_transactions']):>8}   expected= 10 Million")
print(f"{'sku_inventory_flags':20s}   actual={len(df['sku_inventory_flags']):>8}   expected= 600 (200 STOCKOUT_RISK + 400 SLOW_MOVER)")


In [ ]:
print(df['sku_inventory_flags']['flag'].value_counts())

**Master tables `primary key` uniqueness check**

In [ ]:
primary_key = {
    "store_master": "store_id",
    "sku_master": "sku_id",
    'customer_master': "cust_id",
    'promotions': "promo_id",
}

for name, key in primary_key.items():
    duplicates = df[name][key].duplicated().sum()
    print(f"{name:20s}   duplicates={duplicates:>3}")



**Referential integrity**
 > every FK in sales/inventory must exist in its master table

In [ ]:
checks = [
    #(child table) (child key) (parent table) (parent key)
    ("sales_transactions", "store_id", "store_master", "store_id"),
    ("sales_transactions", "sku_id", "sku_master", "sku_id"),
    ("sales_transactions", "customer_id", "customer_master", "cust_id"),
    ("inventory_snapshot", "store_id", "store_master", "store_id"),
    ("inventory_snapshot", "sku_id", "sku_master", "sku_id"),
]

for child_table, child_key, parent_table, parent_key in checks:

  child_values = df[child_table][child_key]
  parent_values = df[parent_table][parent_key]

  valid = child_values.isin(parent_values).all()

  # print(f"{child_table}.{child_key:13s} -> \t {parent_table}.{parent_key}: {'OK' if valid else 'MISMATCH':}")

  child_full = f"{child_table}.{child_key}"
  parent_full = f"{parent_table}.{parent_key}"
  status = 'OK' if valid else 'MISMATCH'

  print(f"{child_full:32s} ->    {parent_full:26s} :    {status}")


`promo_id` is allowed to be null (no promotion applied) — check only the non-null ones

In [ ]:
promo_values = df['sales_transactions']['promo_id'].dropna()

valid = promo_values.isin(df['promotions']['promo_id']).all()
print(f"sales_transactions.promo_id (non-null) -> promotions.promo_id: {'OK' if valid else 'MISMATCH'}")
